# Training ML Models for Loop Unrolling Prediction

This notebook trains machine learning models to predict whether loop unrolling will improve performance.

## Objectives
1. Load and explore the collected dataset
2. Prepare features and labels
3. Train baseline models (Logistic Regression, Decision Tree, Random Forest)
4. Evaluate model performance
5. Analyze feature importance
6. Compare against LLVM's heuristics (if available)

In [ ]:
import sys
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
)

# Setup plotting
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

project_root = Path.cwd().parent

## 1. Load Dataset

In [ ]:
# Load the dataset
dataset_path = project_root / 'data' / 'processed' / 'dataset.csv'

if not dataset_path.exists():
    print(f"❌ Dataset not found at: {dataset_path}")
    print("\nRun data collection first:")
    print("  python src/collect_dataset.py --runs 20")
else:
    df = pd.read_csv(dataset_path)
    print(f"✓ Loaded dataset: {len(df)} samples")
    print(f"  Shape: {df.shape}")
    display(df.head())

## 2. Exploratory Data Analysis

In [ ]:
# Dataset info
print("Dataset Info:")
print("=" * 60)
df.info()

print("\nBasic Statistics:")
print("=" * 60)
display(df.describe())

In [ ]:
# Class distribution
print("\nClass Distribution (Beneficial vs Not):")
print("=" * 60)
class_counts = df['beneficial'].value_counts()
print(f"  Not Beneficial (0): {class_counts.get(0, 0)} ({class_counts.get(0, 0) / len(df) * 100:.1f}%)")
print(f"  Beneficial (1):     {class_counts.get(1, 0)} ({class_counts.get(1, 0) / len(df) * 100:.1f}%)")

# Visualize
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Class distribution
class_counts.plot(kind='bar', ax=ax1, color=['#e74c3c', '#2ecc71'], alpha=0.8)
ax1.set_xlabel('Beneficial')
ax1.set_ylabel('Count')
ax1.set_title('Class Distribution')
ax1.set_xticklabels(['Not Beneficial', 'Beneficial'], rotation=0)

# Speedup distribution
ax2.hist(df['speedup'], bins=30, color='#3498db', alpha=0.7, edgecolor='black')
ax2.axvline(1.0, color='gray', linestyle='--', linewidth=2, label='Baseline (1.0x)')
ax2.axvline(1.05, color='orange', linestyle=':', linewidth=2, label='Threshold (1.05x)')
ax2.set_xlabel('Speedup')
ax2.set_ylabel('Frequency')
ax2.set_title('Speedup Distribution')
ax2.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Feature correlations with target
feature_cols = [
    'num_instructions',
    'num_load_instructions',
    'num_store_instructions',
    'num_branches',
    'num_arithmetic_ops',
    'estimated_trip_count',
    'num_phi_nodes',
]

correlations = df[feature_cols + ['beneficial']].corr()['beneficial'].drop('beneficial').sort_values()

plt.figure(figsize=(10, 6))
correlations.plot(kind='barh', color=correlations.apply(lambda x: '#2ecc71' if x > 0 else '#e74c3c'), alpha=0.8)
plt.xlabel('Correlation with Beneficial')
plt.title('Feature Correlations with Target')
plt.axvline(0, color='black', linewidth=0.8)
plt.tight_layout()
plt.show()

print("\nTop Positive Correlations:")
print(correlations.tail())
print("\nTop Negative Correlations:")
print(correlations.head())

## 3. Prepare Data for Training

In [ ]:
# Select features (exclude metadata and target)
feature_columns = [
    'num_instructions',
    'num_basic_blocks',
    'num_load_instructions',
    'num_store_instructions',
    'num_branches',
    'num_calls',
    'num_arithmetic_ops',
    'estimated_trip_count',
    'has_constant_trip_count',
    'nesting_depth',
    'num_phi_nodes',
    'num_memory_dependencies',
    'has_early_exit',
    'num_exits',
]

X = df[feature_columns].copy()
y = df['beneficial'].copy()

print(f"Features shape: {X.shape}")
print(f"Labels shape: {y.shape}")
print(f"\nFeatures: {list(X.columns)}")

In [ ]:
# Handle missing values (estimated_trip_count may be -1)
print("\nMissing/Invalid values:")
print((X == -1).sum())

# Replace -1 with a large value for "unknown trip count"
X['estimated_trip_count'] = X['estimated_trip_count'].replace(-1, 1000000)

# Check for any NaN
print(f"\nNaN values: {X.isna().sum().sum()}")

In [ ]:
# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set: {len(X_train)} samples")
print(f"Test set: {len(X_test)} samples")
print(f"\nClass distribution in training:")
print(f"  Not Beneficial: {(y_train == 0).sum()}")
print(f"  Beneficial: {(y_train == 1).sum()}")

In [ ]:
# Feature scaling (important for Logistic Regression)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("✓ Features scaled")

## 4. Train Baseline Models

In [ ]:
# Define models
models = {
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000),
    'Decision Tree': DecisionTreeClassifier(random_state=42, max_depth=5),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42, max_depth=10),
}

# Train and evaluate each model
results = {}

for name, model in models.items():
    print(f"\n{'=' * 70}")
    print(f"Training: {name}")
    print(f"{'=' * 70}")
    
    # Use scaled data for Logistic Regression, unscaled for tree-based
    if 'Logistic' in name:
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)
        
        # Cross-validation on scaled data
        cv_scores = cross_val_score(model, X_train_scaled, y_train, cv=5, scoring='accuracy')
    else:
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        
        # Cross-validation on unscaled data
        cv_scores = cross_val_score(model, X_train, y_train, cv=5, scoring='accuracy')
    
    # Metrics
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, zero_division=0)
    recall = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    
    results[name] = {
        'model': model,
        'predictions': y_pred,
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'cv_mean': cv_scores.mean(),
        'cv_std': cv_scores.std(),
    }
    
    print(f"\nTest Set Performance:")
    print(f"  Accuracy:  {accuracy:.3f}")
    print(f"  Precision: {precision:.3f}")
    print(f"  Recall:    {recall:.3f}")
    print(f"  F1-Score:  {f1:.3f}")
    print(f"\nCross-Validation (5-fold):")
    print(f"  Mean Accuracy: {cv_scores.mean():.3f} ± {cv_scores.std():.3f}")
    print(f"\nClassification Report:")
    print(classification_report(y_test, y_pred, target_names=['Not Beneficial', 'Beneficial']))

## 5. Compare Models

In [ ]:
# Model comparison
comparison_df = pd.DataFrame({
    'Model': list(results.keys()),
    'Accuracy': [r['accuracy'] for r in results.values()],
    'Precision': [r['precision'] for r in results.values()],
    'Recall': [r['recall'] for r in results.values()],
    'F1-Score': [r['f1'] for r in results.values()],
    'CV Accuracy': [r['cv_mean'] for r in results.values()],
})

print("\n" + "=" * 70)
print("Model Comparison")
print("=" * 70)
display(comparison_df.style.highlight_max(axis=0, subset=['Accuracy', 'Precision', 'Recall', 'F1-Score', 'CV Accuracy']))

# Plot comparison
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Metrics comparison
comparison_df.set_index('Model')[['Accuracy', 'Precision', 'Recall', 'F1-Score']].plot(
    kind='bar', ax=axes[0], alpha=0.8
)
axes[0].set_ylabel('Score')
axes[0].set_title('Model Performance Comparison')
axes[0].set_ylim([0, 1.1])
axes[0].legend(loc='lower right')
axes[0].grid(axis='y', alpha=0.3)

# CV accuracy
cv_data = [
    (name, r['cv_mean'], r['cv_std']) 
    for name, r in results.items()
]
names, means, stds = zip(*cv_data)
axes[1].bar(names, means, yerr=stds, capsize=5, alpha=0.8, color='#3498db')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Cross-Validation Accuracy (5-fold)')
axes[1].set_ylim([0, 1.1])
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Confusion Matrices

In [ ]:
# Plot confusion matrices for all models
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for idx, (name, result) in enumerate(results.items()):
    cm = confusion_matrix(y_test, result['predictions'])
    
    sns.heatmap(
        cm,
        annot=True,
        fmt='d',
        cmap='Blues',
        ax=axes[idx],
        cbar=False,
        xticklabels=['Not Beneficial', 'Beneficial'],
        yticklabels=['Not Beneficial', 'Beneficial'],
    )
    axes[idx].set_title(f'{name}\n(Accuracy: {result["accuracy"]:.3f})')
    axes[idx].set_ylabel('True Label')
    axes[idx].set_xlabel('Predicted Label')

plt.tight_layout()
plt.show()

## 7. Feature Importance Analysis

In [ ]:
# Feature importance from tree-based models
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Decision Tree
dt_importance = pd.DataFrame({
    'Feature': feature_columns,
    'Importance': results['Decision Tree']['model'].feature_importances_
}).sort_values('Importance', ascending=False)

dt_importance.plot(kind='barh', x='Feature', y='Importance', ax=axes[0], legend=False, color='#e74c3c', alpha=0.8)
axes[0].set_title('Decision Tree Feature Importance')
axes[0].set_xlabel('Importance')

# Random Forest
rf_importance = pd.DataFrame({
    'Feature': feature_columns,
    'Importance': results['Random Forest']['model'].feature_importances_
}).sort_values('Importance', ascending=False)

rf_importance.plot(kind='barh', x='Feature', y='Importance', ax=axes[1], legend=False, color='#2ecc71', alpha=0.8)
axes[1].set_title('Random Forest Feature Importance')
axes[1].set_xlabel('Importance')

plt.tight_layout()
plt.show()

print("\nTop 5 Most Important Features (Random Forest):")
print(rf_importance.head())

## 8. Save Best Model

In [ ]:
import pickle

# Find best model by F1-score
best_model_name = max(results.keys(), key=lambda k: results[k]['f1'])
best_model = results[best_model_name]['model']

print(f"Best model: {best_model_name}")
print(f"  F1-Score: {results[best_model_name]['f1']:.3f}")

# Save model and scaler
model_dir = project_root / 'models'
model_dir.mkdir(exist_ok=True)

model_path = model_dir / 'best_model.pkl'
scaler_path = model_dir / 'scaler.pkl'

with open(model_path, 'wb') as f:
    pickle.dump(best_model, f)

with open(scaler_path, 'wb') as f:
    pickle.dump(scaler, f)

print(f"\n✓ Model saved to: {model_path}")
print(f"✓ Scaler saved to: {scaler_path}")

## 9. Summary & Next Steps

### Key Findings
- Best performing model and its accuracy
- Most important features for prediction
- Common misclassification patterns

### Possible Improvements
1. **Collect more data**: Add more diverse benchmark programs
2. **Feature engineering**: Create derived features (e.g., instruction ratios, memory/compute balance)
3. **Hyperparameter tuning**: Use GridSearchCV to optimize model parameters
4. **Ensemble methods**: Try XGBoost or stacking
5. **Handle class imbalance**: Try SMOTE or class weights if needed
6. **Extract LLVM decisions**: Compare against LLVM's actual unrolling choices

### Deployment
- Integrate with LLVM pass: Use trained model in a custom LLVM optimization pass
- Build prediction API: Serve model via REST API for real-time predictions
- Create CLI tool: Analyze programs and suggest unrolling strategy